# 00 · Organização do Ambiente
### Aula prática — Arquitetura Medalhão com o dataset Olist (Brazilian E-Commerce)

Objetivos deste notebook:
1. Entender a estrutura de diretórios/tabelas que vamos usar (Landing → Bronze → Silver → Gold)
2. Criar o `catalog` e os `schemas` no Unity Catalog
3. Criar o **Volume** que representa a nossa zona de *Landing*
4. Validar que os arquivos `.csv` de origem estão disponíveis

Este notebook é reaproveitado pelos demais (`%run ./00_Organizacao_do_Ambiente`), então evite
alterar os nomes das variáveis abaixo sem ajustar os outros notebooks.

## 1. Parâmetros do ambiente

In [ ]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

## 2. Criação do Catalog, Schemas e Volume

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e volume prontos.")

## 3. Upload dos arquivos de origem

Vamos importar os 6 arquivos que serão utilizados no Databricks, em **Catalog** → `<seu catalog>` → `landing` → `raw_files` → **Upload to this volume**:
   - `olist_customers_dataset.csv`
   - `olist_orders_dataset.csv`
   - `olist_order_items_dataset.csv`
   - `olist_products_dataset.csv`
   - `olist_order_payments_dataset.csv`
   - `product_category_name_translation.csv`

## 4. Validação da Landing Zone

In [ ]:
expected_files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_payments_dataset.csv",
    "product_category_name_translation.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")